# Grab App Reviews Sentiment Analysis

## SVM + TF-IDF


### Import Library

In [ ]:
!pip install -q sastrawi

In [25]:
import pandas as pd
import numpy as np
import csv
import requests
import json
from io import StringIO
import matplotlib.pyplot as plt
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from wordcloud import WordCloud
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load Dataset

In [4]:
url = 'https://raw.githubusercontent.com/bluga404/sa-grab/main/reviews.csv'
reviews = pd.read_csv(url)

# reviews and columns
total_reviews, total_columns  = reviews.shape
print(f"Total reviews: {total_reviews}, Total columns: {total_columns}")
print("-"*50)
reviews.info()

Total reviews: 117000, Total columns: 11
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117000 entries, 0 to 116999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   reviewId              117000 non-null  object
 1   userName              117000 non-null  object
 2   userImage             117000 non-null  object
 3   content               117000 non-null  object
 4   score                 117000 non-null  int64 
 5   thumbsUpCount         117000 non-null  int64 
 6   reviewCreatedVersion  94236 non-null   object
 7   at                    117000 non-null  object
 8   replyContent          31714 non-null   object
 9   repliedAt             31714 non-null   object
 10  appVersion            94236 non-null   object
dtypes: int64(2), object(9)
memory usage: 9.8+ MB


### Preprocessing

In [5]:
# drop nan and duplicates
reviews = reviews.dropna()
reviews = reviews.drop_duplicates()

In [6]:
# stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stem_cache = {}

# stopwords
stopword = set(stopwords.words('indonesian')) \
            | set(stopwords.words('english'))

# regex precompiled
mention = re.compile(r'@[A-Za-z0-9]+')
hashtag = re.compile(r'#[A-Za-z0-9]+')
rt = re.compile(r'\bRT\b')
url = re.compile(r'http\S+')
number = re.compile(r'\d+')
symbol = re.compile(r'[^\w\s]')

slang_url = 'https://raw.githubusercontent.com/okkyibrohim/id-abusive-language-detection/master/kamusalay.csv'
slang_df = pd.read_csv(slang_url, header=None, names=['slang', 'formal'])

slangwords = dict(zip(
    slang_df['slang'].str.lower(),
    slang_df['formal']
))

In [7]:
def cleaningText(text):
    text = mention.sub('', text)
    text = hashtag.sub('', text)
    text = rt.sub('', text)
    text = url.sub('', text)
    text = number.sub('', text)
    text = symbol.sub('', text)
    return text.strip().lower()

def stem_word(word):
    if word in stem_cache:
        return stem_cache[word]

    stemmed = stemmer.stem(word)
    stem_cache[word] = stemmed
    return stemmed

def fixSlang(text):
    # split text to words
    words = text.split()
    # slang + stopword in one pass
    words = [
        stem_word(slangwords.get(w, w))
        for w in words
        if w not in stopword
    ]
    return ' '.join(words)

In [8]:
reviews['final_text'] = reviews['content'].apply(cleaningText)
reviews['final_text'] = reviews['final_text'].apply(fixSlang)
reviews.to_csv('clean_reviews.csv', index=False)
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24020 entries, 3 to 116996
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              24020 non-null  object
 1   userName              24020 non-null  object
 2   userImage             24020 non-null  object
 3   content               24020 non-null  object
 4   score                 24020 non-null  int64 
 5   thumbsUpCount         24020 non-null  int64 
 6   reviewCreatedVersion  24020 non-null  object
 7   at                    24020 non-null  object
 8   replyContent          24020 non-null  object
 9   repliedAt             24020 non-null  object
 10  appVersion            24020 non-null  object
 11  final_text            24020 non-null  object
dtypes: int64(2), object(10)
memory usage: 2.4+ MB


### Labeling

In [ ]:
def load_lexicons():
    # Positive lexicon
    positive_lexicon = {}
    pos_url = 'https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv'
    response = requests.get(pos_url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')
        for row in reader:
            positive_lexicon[row[0]] = int(row[1])
    else:
        raise Exception("Failed to fetch positive lexicon data")

    # Negative lexicon
    negative_lexicon = {}
    neg_url = 'https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv'
    response = requests.get(neg_url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')
        for row in reader:
            negative_lexicon[row[0]] = int(row[1])
    else:
        raise Exception("Failed to fetch negative lexicon data")

    return positive_lexicon, negative_lexicon

def sentiment_label(text, pos_lex, neg_lex):
    score = 0
    words = text.split()

    for w in words:
        score += pos_lex.get(w, 0)
        score += neg_lex.get(w, 0)

    return 1 if score > 0 else -1 if score < 0 else 0

In [10]:
pos_lex, neg_lex = load_lexicons()
reviews['label'] = reviews['final_text'].apply(
    lambda x: sentiment_label(x, pos_lex, neg_lex)
)
print(reviews['label'].value_counts())

label
-1    16913
 1     5798
 0     1309
Name: count, dtype: int64


In [11]:
reviews.to_csv('labeled_reviews.csv', index=False)
reviews = reviews.dropna()

### Data Split and Feature Extraction

In [12]:
X = reviews['final_text']
y = reviews['label']

tfidf = TfidfVectorizer(max_features=5000, min_df=17, max_df=0.8 )
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

### Modeling

In [13]:
svm = LinearSVC(C=1.0, class_weight='balanced', random_state=42)
svm.fit(X_train.toarray(), y_train)

LinearSVC(class_weight='balanced', random_state=42)

In [14]:
y_pred_train_svm = svm.predict(X_train.toarray())
y_pred_test_svm = svm.predict(X_test.toarray())

In [15]:
accuracy_train_svm = accuracy_score(y_pred_train_svm, y_train)
accuracy_test_svm = accuracy_score(y_pred_test_svm, y_test)

In [16]:
print('SVM + TF-IDF - Train accuracy:', accuracy_train_svm)
print('SVM + TF-IDF - Test accuracy:', accuracy_test_svm)

SVM + TF-IDF - Train accuracy: 0.9485547757820864
SVM + TF-IDF - Test accuracy: 0.8974465723008604


### Testing Model / Inference

In [17]:
texts = [
    "Driver grab sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [18]:
X_new = tfidf.transform(texts)
y_pred_new = svm.predict(X_new.toarray())

label_map = {
    -1: "Negative",
     0: "Netral",
     1: "Positive"
}

for text, label in zip(texts, y_pred_new):
    print("Text:", text)
    print("Prediction:", label_map[label])
    print("-" * 50)

Text: Driver grab sangat buruk dan mengecewakan
Prediction: Negative
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive
--------------------------------------------------
Text: saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive
--------------------------------------------------


In [20]:
import joblib
joblib.dump(svm, 'svm_sentiment_model.pkl')

['svm_sentiment_model.pkl']

## BiLSTM + Word2Vec

### Import Libraries

In [22]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 92.3 MB/s eta 0:00:00


In [23]:
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/bluga404/sa-grab/main/labeled_reviews.csv'
reviews = pd.read_csv(url)
reviews = reviews.dropna()
reviews.info()

### Data Split and Feature Extraction

In [26]:
# get text
texts = reviews['final_text']
# convert text to list
texts = texts.tolist()
# apply tokenization for every sentence in text list
tokenized_text = [word_tokenize(sentence) for sentence in texts]

In [27]:
w2v_model = Word2Vec(
    window = 10,
    min_count = 5,
    workers = 4,
    epochs = 10
)

w2v_model.build_vocab(tokenized_text, progress_per=1000)
w2v_model.train(tokenized_text, total_examples=w2v_model.corpus_count, epochs=w2v_model.epochs)
w2v_model.save('word2vec-indo.model')

In [28]:
w2v_model = Word2Vec.load('word2vec-indo.model')
w2v_model.wv.most_similar("lambat")

[('datang', 0.7521166205406189),
 ('undur', 0.7495748400688171),
 ('tunggu', 0.7428390979766846),
 ('proses', 0.6958680748939514),
 ('dapur', 0.682796061038971),
 ('estimasi', 0.674249529838562),
 ('lapar', 0.6741940379142761),
 ('molor', 0.6698213219642639),
 ('perut', 0.6589049696922302),
 ('stengah', 0.6557420492172241)]

In [29]:
print("vocab length:", len(w2v_model.wv.key_to_index))

vocab length: 3404


In [30]:
length = [len(tokens) for tokens in tokenized_text]
print(f"longest: {max(length)} words")
print(f"shortest: {min(length)} words")
print(f"shortest: {np.median(length)} words")

longest: 100 words
shortest: 0 words
shortest: 12.0 words


In [31]:
vocab_size = 10000
max_len = 50
embedding_dim = w2v_model.vector_size

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<oov>")
tokenizer.fit_on_texts(tokenized_text)
word_index = tokenizer.word_index
print(f"there are {len(word_index)} unique tokens")

there are 22860 unique tokens


In [32]:
sequences = tokenizer.texts_to_sequences(tokenized_text)
X_data = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

In [33]:
# +1 cause index 0 reserved for padding
num_words = min(vocab_size, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))

found_words = 0
for word, i in word_index.items():
    if i > vocab_size:
        continue

    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
        found_words += 1
    else:
        pass

print(f"mapped {found_words} from word2vec to keras")

mapped 3404 from word2vec to keras


In [34]:
y = reviews['label'].values
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y,
    test_size = 0.3,
    stratify=y
)

# shift labels to be 0, 1, 2
y_train = y_train + 1
y_test = y_test + 1

In [35]:
print(f"training data shape: {X_train.shape}")
print(f"test data shape: {X_test.shape}")

training data shape: (16814, 50)
test data shape: (7206, 50)


### Modeling

In [36]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        input_length=max_len,
        trainable=True
    )
)

# Bi-LSTM Layer
model.add(Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001))))
model.add(Dropout(0.5))

model.add(Bidirectional(LSTM(32, return_sequences=False, kernel_regularizer=l2(0.001))))
model.add(Dropout(0.5))

# Dense Layer
model.add(Dense(32, activation='relu'))

# Output Layer
model.add(Dense(3, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
optimizer = Adam(learning_rate=1e-4)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [38]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 39s 42ms/step - accuracy: 0.6711 - loss: 1.3106 - val_accuracy: 0.7634 - val_loss: 1.0059
Epoch 2/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.7696 - loss: 0.9485 - val_accuracy: 0.7973 - val_loss: 0.8160
Epoch 3/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.8135 - loss: 0.7780 - val_accuracy: 0.8175 - val_loss: 0.7283
Epoch 4/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.8416 - loss: 0.6639 - val_accuracy: 0.8312 - val_loss: 0.6454
Epoch 5/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.8635 - loss: 0.5764 - val_accuracy: 0.8371 - val_loss: 0.6124
Epoch 6/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.8905 - loss: 0.4890 - val_accuracy: 0.8621 - val_loss: 0.5953
Epoch 7/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.9086 - loss: 0.4297 - val_accuracy: 0.8615 - val_loss: 0.5475
Epoch 8/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.9261 - loss: 0.3693 - val_a

In [39]:
results = model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {results[0]:.4f}")
print(f"Test Accuracy: {results[1]*100:.2f}%")

226/226 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8933 - loss: 0.4444
Test Loss: 0.4362
Test Accuracy: 89.44%


### Testing Model / Inference

In [40]:
texts = [
    "Driver grab sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [41]:
processed_texts = []

for text in texts:
    text = str(text).lower()
    fixed = fixSlang(text)
    processed_texts.append(fixed)

final_tokens = [word_tokenize(t) for t in processed_texts]
seqs = tokenizer.texts_to_sequences(final_tokens)
X_new = pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')

predictions = model.predict(X_new)
y_pred_indices = np.argmax(predictions, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step


In [42]:
label_map = {
    -1: "Negative",
     0: "Netral",
     1: "Positive"
}

for text, index in zip(texts, y_pred_indices):
    # Convert 0,1,2 back to -1,0,1
    original_label = index - 1
    sentiment = label_map[original_label]

    print(f"Text: {text}")
    print(f"Prediction: {sentiment}")
    print("-" * 50)

Text: Driver grab sangat buruk dan mengecewakan
Prediction: Negative
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive
--------------------------------------------------
Text: saya sudah pesan grabfood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Netral
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive
--------------------------------------------------


## IndoBERT-Base

### Import Library

In [43]:
!pip install -U -q transformers
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tf_keras

### Load Dataset

In [44]:
url = 'https://raw.githubusercontent.com/bluga404/sa-grab/main/labeled_reviews.csv'
reviews = pd.read_csv(url)
reviews = reviews.dropna()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24018 entries, 0 to 24019
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              24018 non-null  object
 1   userName              24018 non-null  object
 2   userImage             24018 non-null  object
 3   content               24018 non-null  object
 4   score                 24018 non-null  int64 
 5   thumbsUpCount         24018 non-null  int64 
 6   reviewCreatedVersion  24018 non-null  object
 7   at                    24018 non-null  object
 8   replyContent          24018 non-null  object
 9   repliedAt             24018 non-null  object
 10  appVersion            24018 non-null  object
 11  final_text            24018 non-null  object
 12  label                 24018 non-null  int64 
dtypes: int64(3), object(10)
memory usage: 2.6+ MB


### Data Split and Feature Extraction

In [45]:
X = reviews['final_text'].astype(str).tolist()
y = reviews['label'].values + 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
      stratify=y
)

In [46]:
tokenizer = AutoTokenizer.from_pretrained("sarahlintang/IndoBERT")

def fast_encode(texts, tokenizer, max_len=60):
    inputs = tokenizer(
        texts,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    return dict(inputs)

train_data = fast_encode(X_train, tokenizer)
test_data = fast_encode(X_test, tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


### Modeling

In [47]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "sarahlintang/IndoBERT",
    num_labels=3,
    from_pt=True
)

optimizer = tf_keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
model.summary()

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  112921344 
                                                                 
 dropout_37 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  2307      
                                                                 
Total params: 112923651 (430.77 MB)
Trainable params: 112923651 (430.77 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [48]:
history = model.fit(
    train_data,
    y_train,
    validation_split=0.1,
    epochs=4,
    batch_size=16
)

Epoch 1/4
946/946 [==============================] - 344s 245ms/step - loss: 0.4488 - accuracy: 0.8288 - val_loss: 0.3371 - val_accuracy: 0.8811
Epoch 2/4
946/946 [==============================] - 207s 219ms/step - loss: 0.2814 - accuracy: 0.8993 - val_loss: 0.2720 - val_accuracy: 0.8989
Epoch 3/4
946/946 [==============================] - 205s 217ms/step - loss: 0.1955 - accuracy: 0.9279 - val_loss: 0.2728 - val_accuracy: 0.9025
Epoch 4/4
946/946 [==============================] - 205s 217ms/step - loss: 0.1543 - accuracy: 0.9455 - val_loss: 0.2984 - val_accuracy: 0.9067


In [49]:
test_accuracy, test_loss = model.evaluate(test_data, y_test)

226/226 [==============================] - 31s 137ms/step - loss: 0.3167 - accuracy: 0.8987


### Testing Model / Inference

In [50]:
texts = [
    "Driver gofood sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [52]:
processed_texts = []

for text in texts:
    text = str(text).lower()
    fixed = fixSlang(text)
    processed_texts.append(fixed)

inputs = tokenizer(
    processed_texts,
    max_length=60,
    padding='max_length',
    truncation=True,
    return_tensors='tf'
)

logits = model.predict(dict(inputs)).logits

probabilities = tf.nn.softmax(logits).numpy()
y_pred_indices = np.argmax(probabilities, axis=1)

1/1 [==============================] - 7s 7s/step


In [53]:
label_map = {
    -1: "Negative",
     0: "Neutral",
     1: "Positive"
}

for text, index, probs in zip(texts, y_pred_indices, probabilities):
    original_label = index - 1
    sentiment = label_map[original_label]
    confidence = probs[index] * 100

    print(f"Text: {text}")
    print(f"Prediction: {sentiment} ({confidence:.2f}%)")
    print("-" * 50)

Text: Driver gofood sangat buruk dan mengecewakan
Prediction: Negative (99.89%)
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative (99.95%)
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive (99.52%)
--------------------------------------------------
Text: saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative (99.88%)
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive (99.53%)
--------------------------------------------------


In [54]:
!pip freeze > requirements.txt